# Retail Banking: Account Portfolio & Ledger Transaction Dashboard


In [2]:
# Retail Banking: Account Portfolio & Ledger Transaction Dashboard
import html
import random
import sqlite3
from decimal import Decimal, InvalidOperation

import ipywidgets as w
from IPython.display import display

DB_FILE = "credit_union.db"
ACCOUNT_TYPES = ["Checking", "Savings"]
TXN_TYPES = ["Deposit", "Withdrawal"]
DELETE_MODES = ["Close a Full Customer Profile", "Close a Single Account ID"]


class ValidationError(Exception):
    pass


# ---------------- DATABASE ----------------
conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON")  # enable foreign key checks

with conn:
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS customers (
        customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name        TEXT NOT NULL,
        tax_id      TEXT NOT NULL UNIQUE
    );
    CREATE TABLE IF NOT EXISTS accounts (
        account_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id  INTEGER NOT NULL,
        account_type TEXT NOT NULL CHECK (account_type IN ('Checking','Savings')),
        open_date    TEXT NOT NULL DEFAULT (date('now')),
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ON DELETE CASCADE
    );
    CREATE TABLE IF NOT EXISTS transactions (
        transaction_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        account_id       INTEGER NOT NULL,
        amount           REAL NOT NULL CHECK (amount > 0),
        transaction_type TEXT NOT NULL CHECK (transaction_type IN ('Deposit','Withdrawal')),
        FOREIGN KEY (account_id) REFERENCES accounts(account_id) ON DELETE CASCADE
    );
    """)

# sample data only if the database is empty
if conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 0:
    with conn:
        conn.executemany("INSERT INTO customers(name, tax_id) VALUES (?,?)",
                         [("Aarav Sharma", "TAX-100001"), ("Priya Nair", "TAX-100002"),
                          ("Rohan Mehta", "TAX-100003")])
        conn.executemany("INSERT INTO accounts(customer_id, account_type, open_date) VALUES (?,?,?)",
                         [(1, "Checking", "2024-01-15"), (1, "Savings", "2024-02-01"),
                          (2, "Checking", "2024-03-10"), (3, "Savings", "2024-05-20")])
        conn.executemany("INSERT INTO transactions(account_id, amount, transaction_type) VALUES (?,?,?)",
                         [(1, 5000.00, "Deposit"), (1, 1200.50, "Withdrawal"),
                          (2, 15000.00, "Deposit"), (3, 2500.00, "Deposit"),
                          (3, 300.75, "Withdrawal"), (4, 10000.00, "Deposit")])

VIEWS = {
    "Master Ledger": (
        ["Customer Name", "Account Type", "Amount", "Transaction Type"],
        """SELECT c.name, a.account_type, t.amount, t.transaction_type
           FROM transactions t
           JOIN accounts  a ON t.account_id  = a.account_id
           JOIN customers c ON a.customer_id = c.customer_id
           ORDER BY t.transaction_id"""),
    "Customers": (["customer_id", "name", "tax_id"],
                  "SELECT customer_id, name, tax_id FROM customers"),
    "Accounts": (["account_id", "customer_id", "account_type", "open_date"],
                 "SELECT account_id, customer_id, account_type, open_date FROM accounts"),
    "Transactions": (["transaction_id", "account_id", "amount", "transaction_type"],
                     "SELECT transaction_id, account_id, amount, transaction_type FROM transactions"),
}


# ---------------- LOGIC ----------------
def parse_amount(text):
    text = (text or "").strip()
    if not text:
        raise ValidationError("Transaction amount cannot be blank.")
    try:
        value = Decimal(text)
    except InvalidOperation:
        raise ValidationError("Amount must be a number (e.g. 250 or 99.50).")
    if not value.is_finite():
        raise ValidationError("Amount must be a valid number.")
    if value < 0:
        raise ValidationError("Amount cannot be negative.")
    if value == 0:
        raise ValidationError("Amount must be greater than zero.")
    return float(round(value, 2))


def add_entry(name, account_type, amount_text, txn_type):
    name = (name or "").strip()
    if not name:
        raise ValidationError("Customer name cannot be blank.")
    if not account_type:
        raise ValidationError("Please choose an account type.")
    amount = parse_amount(amount_text)

    with conn:
        row = conn.execute("SELECT customer_id FROM customers WHERE LOWER(name)=LOWER(?)",
                           (name,)).fetchone()
        if row:                                   # existing customer -> reuse ID
            cust_id, status = row[0], "existing customer"
        else:                                     # new customer -> create profile first
            tax = f"TAX-{random.randint(100000, 999999)}"
            cust_id = conn.execute("INSERT INTO customers(name, tax_id) VALUES (?,?)",
                                   (name, tax)).lastrowid
            status = "new customer profile created"
        acc_id = conn.execute("INSERT INTO accounts(customer_id, account_type) VALUES (?,?)",
                              (cust_id, account_type)).lastrowid
        conn.execute("INSERT INTO transactions(account_id, amount, transaction_type) VALUES (?,?,?)",
                     (acc_id, amount, txn_type or "Deposit"))
    return f"Saved: {status} (ID {cust_id}), {account_type} account {acc_id}, {amount:,.2f}."


def delete_record(mode, target_id):
    if target_id is None:
        raise ValidationError("Select a record to delete first.")
    with conn:
        if mode == DELETE_MODES[0]:
            conn.execute("DELETE FROM customers WHERE customer_id=?", (target_id,))
            return f"Customer {target_id} closed; accounts and transactions purged (cascade)."
        conn.execute("DELETE FROM accounts WHERE account_id=?", (target_id,))
        return f"Account {target_id} closed; its transactions purged (cascade)."


def delete_targets(mode):
    if mode == DELETE_MODES[0]:
        rows = conn.execute("SELECT customer_id, name FROM customers").fetchall()
        return [(f"{i} - {n}", i) for i, n in rows]
    rows = conn.execute("""SELECT a.account_id, c.name, a.account_type FROM accounts a
                           JOIN customers c ON a.customer_id = c.customer_id""").fetchall()
    return [(f"{i} - {n} ({t})", i) for i, n, t in rows]


def table_html(view):
    cols, sql = VIEWS[view]
    rows = conn.execute(sql).fetchall()
    head = "".join(f"<th style='padding:4px 12px;background:#e8eef7'>{html.escape(c)}</th>"
                   for c in cols)
    body = "".join("<tr>" + "".join(
        f"<td style='padding:3px 12px;text-align:center'>"
        f"{html.escape(f'{v:,.2f}' if isinstance(v, float) else str(v))}</td>"
        for v in r) + "</tr>" for r in rows)
    return f"<table style='border-collapse:collapse'><tr>{head}</tr>{body}</table>" if rows \
        else "<i>No records.</i>"


# ---------------- GUI ----------------
name = w.Text(description="Name:")
acct = w.Dropdown(options=[""] + ACCOUNT_TYPES, description="Account:")
ttype = w.Dropdown(options=TXN_TYPES, description="Txn Type:")
amt = w.Text(description="Amount:")
add_btn = w.Button(description="Add Record", button_style="success")

mode = w.Dropdown(options=DELETE_MODES, description="Target:", layout=w.Layout(width="420px"))
target = w.Dropdown(description="Record:", layout=w.Layout(width="420px"))
del_btn = w.Button(description="Execute Delete", button_style="danger")
status = w.HTML()

panes = [w.HTML() for _ in VIEWS]
tabs = w.Tab(children=panes)
for i, v in enumerate(VIEWS):
    tabs.set_title(i, v)


def refresh_targets(*_):
    target.options = [("", None)] + delete_targets(mode.value)
    target.value = None


def refresh_all():
    for pane, v in zip(panes, VIEWS):
        pane.value = table_html(v)
    refresh_targets()


def show(msg, ok):
    status.value = f"<b style='color:{'#1a7f37' if ok else '#c62828'}'>{html.escape(msg)}</b>"


def on_add(_):
    try:
        msg = add_entry(name.value, acct.value, amt.value, ttype.value)
    except ValidationError as e:
        show(str(e), False)
        return
    name.value = amt.value = ""          # clear inputs
    acct.value = ""
    ttype.value = "Deposit"              # reset to default
    refresh_all()                        # refresh all 4 tabs
    show(msg, True)


def on_delete(_):
    try:
        msg = delete_record(mode.value, target.value)
    except ValidationError as e:
        show(str(e), False)
        return
    refresh_all()
    show(msg, True)


add_btn.on_click(on_add)
del_btn.on_click(on_delete)
mode.observe(refresh_targets, names="value")
refresh_all()

display(w.VBox([
    w.HTML("<h3>Teller Insert Panel</h3>"), name, acct, ttype, amt, add_btn,
    w.HTML("<h3>Teller Delete Panel</h3>"), mode, target, del_btn, status,
    w.HTML("<h3>Account Dashboard</h3>"), tabs]))